# Custom Model Monitoring: Beyond Built-In Metrics

Snowflake's built-in Model Monitor tracks standard drift and performance metrics (PSI, accuracy, F1, etc.) automatically. But many teams need **custom metrics** that the built-in service doesn't support -- statistical tests, business-specific KPIs, or domain-specific quality checks.

This notebook demonstrates two production-grade approaches for custom model monitoring on Snowflake:

| Approach | How it works | Best for |
|----------|-------------|----------|
| **A: Task + Stored Procedure + Alert** | Python stored proc computes metrics on a warehouse, scheduled by a Task, with an Alert for threshold violations | Lightweight statistical tests, no SPCS infra needed |
| **B: ML Job** | Python script runs on an SPCS compute pool via `submit_file()`, scheduled by a Task | Customized python script/dependencies |

Both approaches follow the same pattern:

```
Compute metric -> Write to CUSTOM_METRICS table -> Alert on threshold
```

### Metrics We'll Implement

| Metric | Type | What it detects |
|--------|------|-----------------|
| **Chi-squared test** | Categorical drift | Distribution shift in categorical features |
| **KS test** | Numerical drift | Distribution shift in numerical features |
| **Prediction confidence shift** | Model behavior | Mean predicted probability drifting over time |

### Prerequisites

Run `00_setup.ipynb` first to create the infrastructure, data, and models.

## 1. Setup and Connect

Import the required libraries and establish a Snowflake session. We import the custom metric functions from `utils/custom_metrics.py`, which contains pure Python+scipy implementations of chi-squared, KS test, and prediction confidence shift. These same functions are reused by the stored procedure and ML Job later.

**Snowsight (Workspaces):** The session is pre-authenticated -- no configuration needed.

**Local development:** Falls back to keypair authentication. Update the `LOCAL CONFIG` section below.

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

# Add utils to path
sys.path.insert(0, "../utils")
from custom_metrics import (
    chi_squared_drift, ks_drift, prediction_confidence_shift, compute_all_metrics
)

In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from cryptography.hazmat.primitives import serialization
    from pathlib import Path

    # ── LOCAL CONFIG (update these for your environment) ──
    ACCOUNT = "<your-account-identifier>"
    USER = "<your-username>"
    ROLE = "ACCOUNTADMIN"
    KEY_PATH = Path.home() / ".snowflake" / "keys" / "rsa_key.p8"
    # ──────────────────────────────────────────────────────

    with open(KEY_PATH, "rb") as f:
        private_key = serialization.load_pem_private_key(f.read(), password=None)

    private_key_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )

    session = Session.builder.configs({
        "account": ACCOUNT,
        "user": USER,
        "private_key": private_key_bytes,
        "role": ROLE,
        "database": "ML_DEMO",
        "schema": "ML_CHURN",
        "warehouse": "ML_CHURN_WH"
    }).create()

print(f"Connected as: {session.get_current_role()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

## 2. Load Shared Data and Model

We load the training data (baseline) and test data from `00_setup.ipynb`, plus retrieve the registered model to generate predictions.

In [ ]:
FEATURE_COLS = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
                "CONTRACT_TYPE", "NUM_SUPPORT_TICKETS", "INTERNET_SERVICE"]
CATEGORICAL_COLS = ["CONTRACT_TYPE", "INTERNET_SERVICE"]
NUMERICAL_COLS = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES", "NUM_SUPPORT_TICKETS"]

# Load baseline (training data) and test data from Snowflake
baseline_data = session.table("ML_DEMO.ML_CHURN.CHURN_TRAIN").to_pandas()
test_data = session.table("ML_DEMO.ML_CHURN.CHURN_TEST").to_pandas()

# Retrieve model V1 for generating predictions
reg = Registry(session=session)
model = reg.get_model("CHURN_MODEL")
mv_v1 = model.version("V1")

print(f"Baseline data: {len(baseline_data)} rows")
print(f"Test data: {len(test_data)} rows")

## 3. Simulate Drift

To make this demo self-contained (no live inference services needed), we create "drifted" current data by perturbing the test set. In production, the "current" data would come from your inference logs or a recent prediction table.

We introduce two types of drift:
- **Numerical drift**: Shift `MONTHLY_CHARGES` upward (simulates price increases)
- **Categorical drift**: Skew `CONTRACT_TYPE` toward month-to-month (category 0)

In [ ]:
np.random.seed(99)
current_data = test_data.copy()

# Numerical drift: shift MONTHLY_CHARGES up by ~$15
current_data["MONTHLY_CHARGES"] = current_data["MONTHLY_CHARGES"] + np.random.normal(15, 5, len(current_data))

# Categorical drift: skew CONTRACT_TYPE toward month-to-month (0)
mask = np.random.random(len(current_data)) < 0.3
current_data.loc[mask, "CONTRACT_TYPE"] = 0

print("Drift injected:")
print(f"  MONTHLY_CHARGES mean: baseline={baseline_data['MONTHLY_CHARGES'].mean():.1f}, "
      f"current={current_data['MONTHLY_CHARGES'].mean():.1f}")
print(f"  CONTRACT_TYPE distribution:")
print(f"    baseline: {baseline_data['CONTRACT_TYPE'].value_counts(normalize=True).sort_index().to_dict()}")
print(f"    current:  {current_data['CONTRACT_TYPE'].value_counts(normalize=True).sort_index().to_dict()}")

In [ ]:
# Generate prediction probabilities for confidence shift metric
# Use batch inference on the registered model
baseline_sp = session.create_dataframe(baseline_data[FEATURE_COLS])
current_sp = session.create_dataframe(current_data[FEATURE_COLS])

baseline_preds = mv_v1.run(baseline_sp, function_name="predict_proba").to_pandas()
current_preds = mv_v1.run(current_sp, function_name="predict_proba").to_pandas()

# Extract probability of churn (class 1)
baseline_probs = baseline_preds.iloc[:, -1].values  # last column = P(churn)
current_probs = current_preds.iloc[:, -1].values

print(f"Baseline P(churn) mean: {baseline_probs.mean():.4f}")
print(f"Current P(churn) mean:  {current_probs.mean():.4f}")

## 4. Compute Metrics Interactively

Before automating anything, let's run the custom metrics locally to see what they produce and verify the drift we injected is detectable.

The `compute_all_metrics()` function runs all three tests in one call:
- **Chi-squared** on each categorical column (CONTRACT_TYPE, INTERNET_SERVICE)
- **KS test** on each numerical column (TENURE_MONTHS, MONTHLY_CHARGES, TOTAL_CHARGES, NUM_SUPPORT_TICKETS)
- **Prediction confidence shift** comparing baseline vs current predicted probabilities

Each result includes the test statistic, p-value, threshold, and a boolean `is_drift` flag. A metric flags drift when the p-value falls below the threshold (0.05) or the absolute shift exceeds the threshold.

In [ ]:
results = compute_all_metrics(
    baseline_df=baseline_data,
    current_df=current_data,
    numerical_cols=NUMERICAL_COLS,
    categorical_cols=CATEGORICAL_COLS,
    baseline_probs=baseline_probs,
    current_probs=current_probs,
    thresholds={"chi_squared": 0.05, "ks_test": 0.05, "prediction_confidence_shift": 0.05}
)

results_df = pd.DataFrame([
    {
        "metric_name": r["metric_name"],
        "statistic": round(r["statistic"], 4),
        "p_value": round(r["p_value"], 6) if r["p_value"] is not None else None,
        "threshold": r["threshold"],
        "is_drift": r["is_drift"],
    }
    for r in results
])

print("Custom Metric Results:")
results_df

**Interpreting the results:**

- **Chi-squared** (p < 0.05): The categorical distribution has shifted significantly. `CONTRACT_TYPE` should show drift since we skewed it. `INTERNET_SERVICE` should not.
- **KS test** (p < 0.05): The numerical distribution has shifted. `MONTHLY_CHARGES` should show drift since we shifted it up. Others should be clean.
- **Prediction confidence shift** (> 0.05): If the drifted features change model outputs, the mean predicted probability shifts.

## 5. Create Results Table

Both approaches (stored procedure and ML Job) write to the same `CUSTOM_METRICS` table. This is the central store for all custom metric history -- each row is a single metric computation at a point in time.

The table schema captures:
- **METRIC_TIMESTAMP**: When the metric was computed
- **METRIC_NAME**: Identifier like `chi_squared__CONTRACT_TYPE` or `ks_test__MONTHLY_CHARGES`
- **METRIC_VALUE**: The test statistic (chi-squared value, KS statistic, or absolute shift)
- **P_VALUE**: Statistical significance (NULL for non-statistical metrics like confidence shift)
- **THRESHOLD / IS_ALERT**: Whether the metric crossed its drift threshold
- **MODEL_NAME / MODEL_VERSION**: Which model the metric applies to
- **DETAILS**: JSON with additional context (observed/expected distributions, means, etc.)

We also write the drifted "current" data to a Snowflake table so the stored procedure can read it.

**Note on SQL-expressible metrics**: If your custom metric can be computed purely in SQL (e.g., `AVG(prediction_probability)`, `STDDEV`, percentile shifts), consider using a Dynamic Table instead. Dynamic Tables auto-refresh based on a target lag, eliminating the need for a Task. This notebook focuses on metrics that require Python (scipy) and therefore need stored procedures or ML Jobs.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE ML_DEMO.ML_CHURN.CUSTOM_METRICS (
    METRIC_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    METRIC_NAME VARCHAR,
    METRIC_VALUE FLOAT,
    P_VALUE FLOAT,
    THRESHOLD FLOAT,
    IS_ALERT BOOLEAN,
    MODEL_NAME VARCHAR,
    MODEL_VERSION VARCHAR,
    DETAILS VARIANT
)
""").collect()
print("Created CUSTOM_METRICS table")

In [ ]:
# Store the drifted data as "current" table for the stored procedure to read
session.write_pandas(current_data, table_name="CHURN_CURRENT", database="ML_DEMO",
                     schema="ML_CHURN", overwrite=True, auto_create_table=True)
print(f"Wrote {len(current_data)} rows to ML_DEMO.ML_CHURN.CHURN_CURRENT")

---

## Approach A: Task + Python Stored Procedure + Alert

This approach runs entirely on a standard warehouse -- no SPCS compute pool required.

**How it works:**
1. A **Python stored procedure** reads baseline and current data, computes metrics using scipy, and writes results to `CUSTOM_METRICS`
2. A **Snowflake Task** calls the procedure on a schedule (e.g., every hour)
3. A **Snowflake Alert** checks the results table and fires when drift is detected

This is the simplest approach and works well for lightweight statistical tests.

### 5a. Create the Python Stored Procedure

The procedure runs on the warehouse and has access to `scipy` via Snowflake's Anaconda channel. It reads data from Snowflake tables, computes all three metrics, and inserts results into `CUSTOM_METRICS`.

Note: Since stored procedures run inside Snowflake, we inline the metric logic rather than importing from `utils/`. The functions are identical.

In [ ]:
session.sql("""
CREATE OR REPLACE PROCEDURE ML_DEMO.ML_CHURN.COMPUTE_CUSTOM_METRICS(
    BASELINE_TABLE VARCHAR,
    CURRENT_TABLE VARCHAR,
    MODEL_NAME VARCHAR,
    MODEL_VERSION VARCHAR
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'scipy', 'numpy', 'pandas')
HANDLER = 'run'
AS
$$
import json
import numpy as np
import pandas as pd
from scipy import stats

def run(session, baseline_table, current_table, model_name, model_version):
    # Configuration
    categorical_cols = ["CONTRACT_TYPE", "INTERNET_SERVICE"]
    numerical_cols = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES", "NUM_SUPPORT_TICKETS"]
    p_threshold = 0.05

    # Read data
    baseline = session.table(baseline_table).to_pandas()
    current = session.table(current_table).to_pandas()

    results = []

    # Chi-squared tests on categorical columns
    for col in categorical_cols:
        baseline_counts = baseline[col].value_counts()
        current_counts = current[col].value_counts()
        all_cats = sorted(set(baseline_counts.index) | set(current_counts.index))
        observed = np.array([current_counts.get(c, 0) for c in all_cats], dtype=float)
        expected_raw = np.array([baseline_counts.get(c, 0) for c in all_cats], dtype=float)
        expected = expected_raw * (observed.sum() / expected_raw.sum())
        expected = np.where(expected == 0, 1e-10, expected)
        stat, p_val = stats.chisquare(observed, f_exp=expected)
        results.append({
            "metric_name": f"chi_squared__{col}",
            "metric_value": float(stat),
            "p_value": float(p_val),
            "threshold": p_threshold,
            "is_alert": bool(p_val < p_threshold),
            "details": {"observed": observed.tolist(), "expected": expected.tolist()}
        })

    # KS tests on numerical columns
    for col in numerical_cols:
        stat, p_val = stats.ks_2samp(
            baseline[col].dropna().values.astype(float),
            current[col].dropna().values.astype(float)
        )
        results.append({
            "metric_name": f"ks_test__{col}",
            "metric_value": float(stat),
            "p_value": float(p_val),
            "threshold": p_threshold,
            "is_alert": bool(p_val < p_threshold),
            "details": {
                "baseline_mean": float(baseline[col].mean()),
                "current_mean": float(current[col].mean())
            }
        })

    # Write results
    for r in results:
        session.sql(f\"\"\"
            INSERT INTO ML_DEMO.ML_CHURN.CUSTOM_METRICS
                (METRIC_NAME, METRIC_VALUE, P_VALUE, THRESHOLD, IS_ALERT, MODEL_NAME, MODEL_VERSION, DETAILS)
            SELECT
                '{r["metric_name"]}',
                {r["metric_value"]},
                {r["p_value"]},
                {r["threshold"]},
                {str(r["is_alert"]).upper()},
                '{model_name}',
                '{model_version}',
                PARSE_JSON('{json.dumps(r["details"])}')
        \"\"\").collect()

    drift_count = sum(1 for r in results if r["is_alert"])
    return f"Computed {len(results)} metrics. {drift_count} drift alerts."
$$
""").collect()
print("Created stored procedure COMPUTE_CUSTOM_METRICS")

### 5b. Test the Procedure

Call the stored procedure manually to verify it works end-to-end. It reads baseline data from `CHURN_TRAIN`, current data from `CHURN_CURRENT`, computes all metrics, and inserts results into `CUSTOM_METRICS`. The return value reports how many metrics were computed and how many triggered drift alerts.

In [ ]:
# Call the stored procedure
result = session.sql("""
CALL ML_DEMO.ML_CHURN.COMPUTE_CUSTOM_METRICS(
    'ML_DEMO.ML_CHURN.CHURN_TRAIN',
    'ML_DEMO.ML_CHURN.CHURN_CURRENT',
    'CHURN_MODEL',
    'V1'
)
""").collect()
print(result[0][0])

In [ ]:
# Verify results
metrics = session.sql("""
SELECT METRIC_TIMESTAMP, METRIC_NAME, ROUND(METRIC_VALUE, 4) AS METRIC_VALUE,
       ROUND(P_VALUE, 6) AS P_VALUE, THRESHOLD, IS_ALERT, MODEL_NAME, MODEL_VERSION
FROM ML_DEMO.ML_CHURN.CUSTOM_METRICS
ORDER BY METRIC_TIMESTAMP DESC, METRIC_NAME
LIMIT 20
""").to_pandas()

print("Results in CUSTOM_METRICS table:")
metrics

### 5c. Schedule with a Task

The Task runs the procedure on a schedule. In production you'd use `SCHEDULE = '1 hour'` or a cron expression. For the demo we create it but leave it suspended.

In [ ]:
session.sql("""
CREATE OR REPLACE TASK ML_DEMO.ML_CHURN.CUSTOM_METRICS_TASK
    WAREHOUSE = ML_CHURN_WH
    SCHEDULE = '60 MINUTE'
    COMMENT = 'Compute custom drift metrics every hour'
AS
    CALL ML_DEMO.ML_CHURN.COMPUTE_CUSTOM_METRICS(
        'ML_DEMO.ML_CHURN.CHURN_TRAIN',
        'ML_DEMO.ML_CHURN.CHURN_CURRENT',
        'CHURN_MODEL',
        'V1'
    )
""").collect()
print("Created task CUSTOM_METRICS_TASK (suspended by default)")

# To start the task in production, run:
# session.sql("ALTER TASK ML_DEMO.ML_CHURN.CUSTOM_METRICS_TASK RESUME").collect()

### 5d. Create an Alert

The Alert checks the `CUSTOM_METRICS` table for recent drift detections. When drift is found, it logs the alert to a history table. In production, you'd replace the action with `SYSTEM$SEND_EMAIL(...)` or a webhook notification via `SYSTEM$SEND_SNOWFLAKE_NOTIFICATION()`.

In [ ]:
# Create alert history table
session.sql("""
CREATE OR REPLACE TABLE ML_DEMO.ML_CHURN.CUSTOM_ALERT_HISTORY (
    ALERT_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    ALERT_MESSAGE VARCHAR,
    METRICS_TRIGGERED VARIANT
)
""").collect()

# Create the alert
session.sql("""
CREATE OR REPLACE ALERT ML_DEMO.ML_CHURN.CUSTOM_DRIFT_ALERT
    WAREHOUSE = ML_CHURN_WH
    SCHEDULE = '60 MINUTE'
    IF (EXISTS (
        SELECT 1 FROM ML_DEMO.ML_CHURN.CUSTOM_METRICS
        WHERE IS_ALERT = TRUE
          AND METRIC_TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
    ))
    THEN
        INSERT INTO ML_DEMO.ML_CHURN.CUSTOM_ALERT_HISTORY (ALERT_MESSAGE, METRICS_TRIGGERED)
        SELECT
            'Drift detected in ' || COUNT(*) || ' metrics',
            ARRAY_AGG(OBJECT_CONSTRUCT(
                'metric', METRIC_NAME,
                'value', METRIC_VALUE,
                'p_value', P_VALUE
            ))
        FROM ML_DEMO.ML_CHURN.CUSTOM_METRICS
        WHERE IS_ALERT = TRUE
          AND METRIC_TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
""").collect()
print("Created alert CUSTOM_DRIFT_ALERT (suspended by default)")

# To activate the alert in production, run:
# session.sql("ALTER ALERT ML_DEMO.ML_CHURN.CUSTOM_DRIFT_ALERT RESUME").collect()

### 5e. Test the Alert Manually

`EXECUTE ALERT` triggers the alert condition check immediately (outside its normal schedule). If drift rows exist in `CUSTOM_METRICS` from the last 2 hours, the action inserts a summary into `CUSTOM_ALERT_HISTORY`.

In production, you would replace the INSERT action with `SYSTEM$SEND_EMAIL(...)` or `SYSTEM$SEND_SNOWFLAKE_NOTIFICATION()` to send email or webhook notifications when drift is detected.

In [ ]:
# Manually execute the alert to test it
session.sql("EXECUTE ALERT ML_DEMO.ML_CHURN.CUSTOM_DRIFT_ALERT").collect()

# Check alert history
alert_history = session.sql("""
SELECT * FROM ML_DEMO.ML_CHURN.CUSTOM_ALERT_HISTORY
ORDER BY ALERT_TIMESTAMP DESC
""").to_pandas()

print("Alert history:")
alert_history

---

## Approach B: ML Job (SPCS)

This approach runs a Python script on a Snowpark Container Services compute pool via `submit_file()`. It's more powerful than the stored procedure approach -- you get a full Python environment with any library, GPU support if needed, and no warehouse size limitations.

**How it works:**
1. A **Python script** connects to Snowflake, reads data, computes metrics, and writes results to `CUSTOM_METRICS`
2. `submit_file()` submits the script to an SPCS compute pool
3. A **Task** can wrap the submission to schedule it
4. Same **Alert** from Approach A watches the results table

**When to use this over Approach A:**
- Your metric computation is too heavy for a warehouse (e.g., training a drift detection model)
- You need custom Python packages not available in Snowflake's Anaconda channel
- You need GPU for computation
- Your computation takes longer than the stored procedure timeout

### 6a. Write the Job Script

The job script is a standalone Python file that will execute on the compute pool. It imports the same metric functions from `custom_metrics.py`.

In [ ]:
JOB_SCRIPT = '''
"""custom_metrics_job.py -- ML Job for computing custom drift metrics."""

import json
import sys
import numpy as np
import pandas as pd
from scipy import stats
from snowflake.snowpark import Session

# The ML Job runtime provides a pre-configured session
def get_session():
    return Session.builder.getOrCreate()

def chi_squared_test(baseline, current, col):
    baseline_counts = baseline[col].value_counts()
    current_counts = current[col].value_counts()
    all_cats = sorted(set(baseline_counts.index) | set(current_counts.index))
    observed = np.array([current_counts.get(c, 0) for c in all_cats], dtype=float)
    expected_raw = np.array([baseline_counts.get(c, 0) for c in all_cats], dtype=float)
    expected = expected_raw * (observed.sum() / expected_raw.sum())
    expected = np.where(expected == 0, 1e-10, expected)
    stat, p_val = stats.chisquare(observed, f_exp=expected)
    return float(stat), float(p_val)

def ks_test(baseline, current, col):
    stat, p_val = stats.ks_2samp(
        baseline[col].dropna().values.astype(float),
        current[col].dropna().values.astype(float)
    )
    return float(stat), float(p_val)

def main():
    session = get_session()

    categorical_cols = ["CONTRACT_TYPE", "INTERNET_SERVICE"]
    numerical_cols = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES", "NUM_SUPPORT_TICKETS"]
    p_threshold = 0.05

    baseline = session.table("ML_DEMO.ML_CHURN.CHURN_TRAIN").to_pandas()
    current = session.table("ML_DEMO.ML_CHURN.CHURN_CURRENT").to_pandas()

    results = []

    for col in categorical_cols:
        stat, p_val = chi_squared_test(baseline, current, col)
        results.append({
            "metric_name": f"chi_squared__{col}",
            "metric_value": stat,
            "p_value": p_val,
            "threshold": p_threshold,
            "is_alert": p_val < p_threshold,
        })

    for col in numerical_cols:
        stat, p_val = ks_test(baseline, current, col)
        results.append({
            "metric_name": f"ks_test__{col}",
            "metric_value": stat,
            "p_value": p_val,
            "threshold": p_threshold,
            "is_alert": p_val < p_threshold,
        })

    for r in results:
        session.sql(f"""
            INSERT INTO ML_DEMO.ML_CHURN.CUSTOM_METRICS
                (METRIC_NAME, METRIC_VALUE, P_VALUE, THRESHOLD, IS_ALERT, MODEL_NAME, MODEL_VERSION, DETAILS)
            SELECT
                \\'{r["metric_name"]}\\',
                {r["metric_value"]},
                {r["p_value"]},
                {r["threshold"]},
                {str(r["is_alert"]).upper()},
                \\'CHURN_MODEL\\',
                \\'V1\\',
                NULL
        """).collect()

    drift_count = sum(1 for r in results if r["is_alert"])
    print(f"Computed {len(results)} metrics. {drift_count} drift alerts.")

if __name__ == "__main__":
    main()
'''

# Write the job script to a local file
from pathlib import Path
job_path = Path("../utils/custom_metrics_job.py")
job_path.write_text(JOB_SCRIPT.strip())
print(f"Job script written to {job_path.resolve()}")

### 6b. Submit the ML Job

`submit_file()` uploads and runs the script on the specified compute pool. You need an existing SPCS compute pool. If you don't have one, skip this section -- Approach A gives you the same results.

In [ ]:
from snowflake.ml.jobs import submit_file

# Change COMPUTE_POOL to match your account's available pool
COMPUTE_POOL = "ML_ONLINE_CPU_POOL"

job = submit_file(
    session=session,
    file_path="../utils/custom_metrics_job.py",
    compute_pool=COMPUTE_POOL,
    stage_name="ML_DEMO.ML_CHURN.CUSTOM_METRICS_STAGE",
)
print(f"Submitted ML Job: {job.id}")
print(f"Status: {job.status}")

In [ ]:
# Wait for the job to complete and check logs
import time

for _ in range(30):  # wait up to 5 minutes
    status = job.status
    print(f"Job status: {status}")
    if status in ("DONE", "FAILED"):
        break
    time.sleep(10)

print("\nJob logs:")
print(job.get_logs())

### 6c. Verify Results

The ML Job writes to the same `CUSTOM_METRICS` table as the stored procedure.

In [ ]:
# Check latest metrics (should include results from both approaches if both were run)
latest = session.sql("""
SELECT METRIC_TIMESTAMP, METRIC_NAME, ROUND(METRIC_VALUE, 4) AS METRIC_VALUE,
       ROUND(P_VALUE, 6) AS P_VALUE, IS_ALERT
FROM ML_DEMO.ML_CHURN.CUSTOM_METRICS
ORDER BY METRIC_TIMESTAMP DESC
LIMIT 20
""").to_pandas()

print("Latest metrics in CUSTOM_METRICS:")
latest

### 6d. Schedule via Task

To schedule the ML Job, create a Task that calls a stored procedure wrapping `submit_file()`. This is a common pattern for recurring ML Jobs.

In [ ]:
# Create a wrapper procedure for submitting the ML Job
session.sql("""
CREATE OR REPLACE PROCEDURE ML_DEMO.ML_CHURN.SUBMIT_CUSTOM_METRICS_JOB()
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-ml-python', 'snowflake-snowpark-python')
HANDLER = 'run'
AS
$$
def run(session):
    from snowflake.ml.jobs import submit_file
    job = submit_file(
        session=session,
        file_path="@ML_DEMO.ML_CHURN.CUSTOM_METRICS_STAGE/custom_metrics_job.py",
        compute_pool="ML_ONLINE_CPU_POOL",
        stage_name="ML_DEMO.ML_CHURN.CUSTOM_METRICS_STAGE",
    )
    return f"Submitted job: {job.id}"
$$
""").collect()

# Create a task to run the ML Job on a schedule
session.sql("""
CREATE OR REPLACE TASK ML_DEMO.ML_CHURN.CUSTOM_METRICS_JOB_TASK
    WAREHOUSE = ML_CHURN_WH
    SCHEDULE = '60 MINUTE'
    COMMENT = 'Submit custom metrics ML Job every hour'
AS
    CALL ML_DEMO.ML_CHURN.SUBMIT_CUSTOM_METRICS_JOB()
""").collect()
print("Created task CUSTOM_METRICS_JOB_TASK (suspended by default)")

# To start: ALTER TASK ML_DEMO.ML_CHURN.CUSTOM_METRICS_JOB_TASK RESUME

---

## 7. Query Metric History

Once the Task + Alert is running in production, the `CUSTOM_METRICS` table accumulates time-series data you can query and visualize.

The first query shows a **drift summary per metric** -- how many runs detected drift, and the average statistic/p-value across all runs. This tells you which features are consistently drifting.

The second query shows a **time-series trend** bucketed by hour. In production (with the Task running hourly), this gives you a chart-ready view of how each metric evolves over time. The `ANY_DRIFT` column flags hours where at least one computation detected drift.

In [ ]:
# Summary: which metrics triggered drift?
drift_summary = session.sql("""
SELECT METRIC_NAME,
       COUNT(*) AS total_runs,
       SUM(CASE WHEN IS_ALERT THEN 1 ELSE 0 END) AS drift_count,
       ROUND(AVG(METRIC_VALUE), 4) AS avg_statistic,
       ROUND(AVG(P_VALUE), 6) AS avg_p_value
FROM ML_DEMO.ML_CHURN.CUSTOM_METRICS
GROUP BY METRIC_NAME
ORDER BY drift_count DESC, METRIC_NAME
""").to_pandas()

print("Drift summary by metric:")
drift_summary

In [ ]:
# Metric trend over time (useful once you have multiple runs)
trend = session.sql("""
SELECT DATE_TRUNC('hour', METRIC_TIMESTAMP) AS hour,
       METRIC_NAME,
       ROUND(AVG(METRIC_VALUE), 4) AS avg_value,
       BOOLOR_AGG(IS_ALERT) AS any_drift
FROM ML_DEMO.ML_CHURN.CUSTOM_METRICS
GROUP BY 1, 2
ORDER BY 1, 2
""").to_pandas()

print("Metric trend (hourly buckets):")
trend

## 8. Cleanup

Remove all objects created by this notebook: tasks, alerts, stored procedures, tables, and stages. Shared resources from `00_setup.ipynb` (database, schema, warehouse, model, train/test tables) are left intact so other demo notebooks can still run.

Tasks and alerts must be suspended before they can be dropped.

In [ ]:
# Suspend tasks and alerts before dropping
session.sql("ALTER TASK IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_METRICS_TASK SUSPEND").collect()
session.sql("ALTER TASK IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_METRICS_JOB_TASK SUSPEND").collect()
session.sql("ALTER ALERT IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_DRIFT_ALERT SUSPEND").collect()

# Drop objects
session.sql("DROP TASK IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_METRICS_TASK").collect()
session.sql("DROP TASK IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_METRICS_JOB_TASK").collect()
session.sql("DROP ALERT IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_DRIFT_ALERT").collect()
session.sql("DROP PROCEDURE IF EXISTS ML_DEMO.ML_CHURN.COMPUTE_CUSTOM_METRICS(VARCHAR, VARCHAR, VARCHAR, VARCHAR)").collect()
session.sql("DROP PROCEDURE IF EXISTS ML_DEMO.ML_CHURN.SUBMIT_CUSTOM_METRICS_JOB()").collect()
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_METRICS").collect()
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_ALERT_HISTORY").collect()
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_CURRENT").collect()
session.sql("DROP STAGE IF EXISTS ML_DEMO.ML_CHURN.CUSTOM_METRICS_STAGE").collect()

print("All custom monitoring objects cleaned up.")

In [ ]:
session.close()
print("Session closed.")